<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/studio/2026-09-23-Cleaning-Clinic-Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# TODO
print(df.shape)
df.info()
print(df.isna().sum())
print('exact duplicates:', df.duplicated().sum())

(8, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  8 non-null      int64  
 1   item      7 non-null      object 
 2   category  8 non-null      object 
 3   qty       7 non-null      float64
 4   price     8 non-null      object 
 5   ts        7 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
exact duplicates: 1


**What is wrong with this data?** List at least five specific problems:

1. Order 1 is a duplicate row so it would be incorrectly counted twice.
2. The price column is text (some values have a dollar sign) so it can't be summed as a number
3. Order 3 has no quantity, so we don't know how many were sold or what it brought in
4. Category names disagree in case and punctuation (Food/food, RainGear/rain-gear)
5. Item names disagree (Cheeseburger/cheese burger, Rain Poncho/rain poncho)
6. Timestamps come in mixed formats

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

# TODO:
log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
# TODO:
clean['price'] = clean['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)

assert clean['price'].dtype == float
# TODO:  -- note that price arrived as text
log('price', 'price arrived as text; removed $ and spaces, converted to float', len(clean))

[price] price arrived as text; removed $ and spaces, converted to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
# TODO:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately
clean = clean[clean['qty'].notna()].copy()
log('qty missing', 'dropped row with no quantity (order 3); units unknown', missing)
clean = clean[clean['qty'] > 0].copy()
log('qty refund', 'excluded refund row (order 5) to report gross sales', negative)

[qty missing] dropped row with no quantity (order 3); units unknown (1 row(s))
[qty refund] excluded refund row (order 5) to report gross sales (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:
print('before:', sorted(clean['category'].unique()))
n_before = clean['category'].nunique()

# TODO: lowercase, strip, remove punctuation
clean['category'] = clean['category'].str.lower().str.strip().str.replace('-', '', regex=False)

# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {'apparel': 'merch'}

print('after: ', sorted(clean['category'].unique()))
log('category', f'normalized case/punctuation; mapped apparel -> merch ({n_before} -> {clean["category"].nunique()})', n_before)

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'raingear']
[category] normalized case/punctuation; mapped apparel -> merch (5 -> 4) (5 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
# TODO
blank_items = clean['item'].isna().sum()

clean['item'] = clean['item'].str.strip()

ITEM_MAP = {'cheese burger': 'Cheeseburger', 'rain poncho': 'Rain Poncho'}
clean['item'] = clean['item'].replace(ITEM_MAP)
clean['item'] = clean['item'].fillna('Unknown')

print(sorted(clean['item'].unique()))
log('item', 'trimmed spaces; mapped spelling variants to one name', len(clean))
log('item blank', 'kept order 7 as "Unknown"; it is a real $12 merch sale', blank_items)

['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt', 'Unknown']
[item] trimmed spaces; mapped spelling variants to one name (5 row(s))
[item blank] kept order 7 as "Unknown"; it is a real $12 merch sale (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [9]:
# TODO
clean['ts'] = pd.to_datetime(clean['ts'], format='mixed', errors='coerce')
failed = clean['ts'].isna().sum()
print('NaT:', failed)

clean['hour'] = clean['ts'].dt.hour
log('ts', 'parsed mixed formats (US month/day); kept NaT row in totals, not usable for hourly', failed)

NaT: 1
[ts] parsed mixed formats (US month/day); kept NaT row in totals, not usable for hourly (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [10]:
# TODO: assertions
assert clean.duplicated().sum() == 0
assert clean['price'].dtype == float
assert clean['qty'].min() >= 1
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])
assert (clean['category'] == clean['category'].str.lower()).all()
assert clean['item'].notna().all()

# TODO: clean['revenue'] = ...
clean['revenue'] = clean['qty'] * clean['price']

# TODO: print rows, units, revenue, distinct categories
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 5
units: 10.0
revenue: 106.5
distinct categories: 4


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [11]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,"price arrived as text; removed $ and spaces, c...",7
2,qty missing,dropped row with no quantity (order 3); units ...,1
3,qty refund,excluded refund row (order 5) to report gross ...,1
4,category,normalized case/punctuation; mapped apparel ->...,5
5,item,trimmed spaces; mapped spelling variants to on...,5
6,item blank,"kept order 7 as ""Unknown""; it is a real $12 me...",1
7,ts,parsed mixed formats (US month/day); kept NaT ...,1


### Step 8 — the decision log
**The decision that mattered most:** The decision that mattered the most by moving revenue total the most was excluding the refund row. One entry (order 5) had a -3 quantity for ponchos (that were 6 dollars a piece).

**Revenue with it:** 106.50 dollars  **Revenue without it:** 88.50 dollars

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
rows_after = len(clean)            # TODO
revenue_after = clean['revenue'].sum()         # TODO
biggest_decision = 'excluded row with negative quantity (refund row)'    # TODO: which choice moved the number most
revenue_other_way = 88.5     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 106.5
decision that mattered: excluded row with negative quantity (refund row)
revenue the other way: 88.5
